<a href="https://colab.research.google.com/github/Pedro4Albuquerque/Processamento-de-Linguagem-Natural/blob/main/Projeto_Integrado_PLN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Projeto Integrado de PLN — ISW037

### Pedro Henrique Albuquerque Souza ###

Este notebook aplica o pipeline completo de PLN ao corpus fixo da atividade:

1. Tokenização  
2. Stopwords, Stemming e Lematização  
3. Análise Sintática  
4. Interpretação Semântica  
5. Extração de Features  
6. Descoberta de Conhecimento em Textos (KDT)  
7. Conclusão Integradora



## Preparação do ambiente

Será utilizado o modelo **`pt_core_news_md`** do spaCy porque, além de tokenização, POS Tagging, dependências e NER, ele possui vetores de palavras necessários para o cálculo de similaridade semântica.


In [1]:

!pip -q install spacy nltk scikit-learn pandas
!python -m spacy download pt_core_news_md


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 17.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:

import spacy
import nltk
import numpy as np
import pandas as pd

from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

nltk.download("rslp", quiet=True)

nlp = spacy.load("pt_core_news_md")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 120)

print("Modelo carregado:", nlp.meta["name"])


Modelo carregado: core_news_md


## Corpus fixo

In [3]:

corpus = [
    "Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.",
    "Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?",
    "Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana.",
    "Qual é o valor do condomínio e do IPTU do apartamento no centro de Araraquara?",
    "A casa na Vila Mariana ainda está disponível para locação?",
    "Aceitam fiador ou seguro fiança para o contrato de aluguel?",
    "Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.",
    "O apartamento é novo ou já foi reformado recentemente?",
    "Qual o prazo médio de aprovação do financiamento bancário?",
    "Estou buscando um imóvel próximo ao Shopping Iguatemi, para compra à vista.",
    "O sobrado no bairro Santa Cecília aceita animais de estimação?",
    "Poderiam enviar mais fotos do apartamento e da planta baixa?",
    "Qual é a taxa de juros aplicada pelo Banco do Brasil nesse tipo de financiamento?",
    "A casa alugada tem vaga de garagem coberta?",
    "Tenho interesse em investir em um imóvel para locação de longo prazo.",
    "Poderia me indicar imóveis disponíveis na região central de Matão?",
]

docs = [nlp(texto) for texto in corpus]

print("Quantidade de documentos:", len(corpus))


Quantidade de documentos: 16



# 1. Tokenização

Tokenização é a divisão do texto em unidades menores chamadas *tokens*.  
Abaixo todos os documentos são processados pelo spaCy e são mostrados, como exemplo, os documentos 1 e 2.


In [4]:

tokens_corpus = [[token.text for token in doc] for doc in docs]

for indice in [0, 1]:
    print(f"Documento {indice + 1}:")
    print(corpus[indice])
    print("\nTokens:")
    print(tokens_corpus[indice])
    print("-" * 100)


Documento 1:
Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.

Tokens:
['Bom', 'dia', '!', 'Meu', 'nome', 'é', 'Marcos', 'Ferreira', 'e', 'tenho', 'interesse', 'no', 'apartamento', 'anunciado', 'na', 'Rua', 'Voluntários', 'da', 'Pátria', ',', 'em', 'Matão', '.']
----------------------------------------------------------------------------------------------------
Documento 2:
Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?

Tokens:
['Vocês', 'trabalham', 'com', 'financiamento', 'pela', 'Caixa', 'Econômica', 'Federal', 'para', 'imóveis', 'usados', '?']
----------------------------------------------------------------------------------------------------



**Comentário:** o spaCy preserva pontuação como tokens separados e identifica corretamente palavras acentuadas e nomes compostos. Isso é importante porque as próximas etapas dependem dessa segmentação inicial.



# 2. Stopwords, Stemming e Lematização

Para esta etapa foi escolhido o **Documento 1**. Primeiro são removidos sinais de pontuação e *stopwords*.  
Depois é aplicado:

- **Stemming:** algoritmo RSLP do NLTK.
- **Lematização:** lema produzido pelo spaCy.


In [5]:

stemmer = RSLPStemmer()
doc_escolhido = docs[0]

comparacao = []

for token in doc_escolhido:
    if token.is_space or token.is_punct or token.is_stop:
        continue

    palavra = token.text
    normalizada = token.lower_
    stem = stemmer.stem(normalizada)
    lema = token.lemma_.lower()

    comparacao.append({
        "Palavra original": palavra,
        "Normalizada": normalizada,
        "Stemming (RSLP)": stem,
        "Lematização (spaCy)": lema
    })

df_comparacao = pd.DataFrame(comparacao)
df_comparacao


,Palavra original,Normalizada,Stemming (RSLP),Lematização (spaCy)
0,dia,dia,dia,dia
1,nome,nome,nom,nome
2,Marcos,marcos,marc,marcos
3,Ferreira,ferreira,ferr,ferreira
4,interesse,interesse,inter,interesse
5,apartamento,apartamento,apart,apartamento
6,anunciado,anunciado,anunci,anunciar
7,Rua,rua,rua,rua
8,Voluntários,voluntários,voluntári,voluntários
9,Pátria,pátria,pátr,pátria



**Comentário:** o stemming tende a reduzir as palavras de forma mais agressiva, podendo gerar radicais que não são palavras completas. Já a lematização procura retornar a forma de dicionário da palavra, mantendo melhor seu significado linguístico.



# 3. Análise Sintática

Foram escolhidos:

- **Documento 3:** “Gostaria de agendar uma visita...”
- **Documento 5:** “A casa na Vila Mariana ainda está disponível para locação?”

O segundo documento é interessante porque contém uma construção copulativa com o verbo **estar**.


In [6]:

def tabela_pos(doc):
    return pd.DataFrame([
        {
            "Token": token.text,
            "POS": token.pos_,
            "Tag": token.tag_,
            "Dependência": token.dep_,
            "Cabeça": token.head.text
        }
        for token in doc
        if not token.is_space
    ])

def extrair_sintagmas(doc):
    # Sintagmas nominais reconhecidos pelo parser do spaCy
    sintagmas_nominais = [chunk.text for chunk in doc.noun_chunks]

    # Raiz sintática
    root = next(token for token in doc if token.dep_ == "ROOT")

    # Heurística para o sintagma verbal:
    # pega a subárvore da raiz e remove a subárvore do sujeito.
    tokens_sujeito = set()
    for token in doc:
        if token.dep_.startswith("nsubj"):
            tokens_sujeito.update(token.subtree)

    tokens_sv = [
        token for token in root.subtree
        if token not in tokens_sujeito and not token.is_punct
    ]

    tokens_sv = sorted(tokens_sv, key=lambda t: t.i)
    sintagma_verbal = " ".join(token.text for token in tokens_sv)

    return sintagmas_nominais, sintagma_verbal, root

def comentar_root(root):
    possui_copula = any(filho.dep_ == "cop" for filho in root.children)

    if possui_copula:
        return (
            f"A ROOT é '{root.text}' ({root.pos_}). "
            "Há uma cópula ligada à raiz, portanto é uma construção copulativa."
        )

    if root.pos_ in {"VERB", "AUX"}:
        return (
            f"A ROOT é '{root.text}' ({root.pos_}). "
            "Neste caso, a raiz funciona como verbo principal da oração."
        )

    return (
        f"A ROOT é '{root.text}' ({root.pos_}). "
        "A raiz não é um verbo lexical comum; isso merece atenção na análise da estrutura."
    )


In [7]:

indices_sintaxe = [2, 4]

for indice in indices_sintaxe:
    doc = docs[indice]

    print("=" * 110)
    print(f"DOCUMENTO {indice + 1}")
    print(corpus[indice])
    print("\nPOS TAGGING:")
    display(tabela_pos(doc))

    sns, sv, root = extrair_sintagmas(doc)

    print("Sintagma(s) Nominal(is) - SN:")
    for sn in sns:
        print(" -", sn)

    print("\nSintagma Verbal - SV:")
    print(" -", sv)

    print("\nROOT:")
    print(" -", comentar_root(root))


DOCUMENTO 3
Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana.

POS TAGGING:


,Token,POS,Tag,Dependência,Cabeça
0,Gostaria,VERB,VERB,ROOT,Gostaria
1,de,SCONJ,SCONJ,mark,agendar
2,agendar,VERB,VERB,xcomp,Gostaria
3,uma,DET,DET,det,visita
4,visita,NOUN,NOUN,obj,agendar
5,ao,ADP,ADP,case,imóvel
6,imóvel,NOUN,NOUN,nmod,visita
7,do,ADP,ADP,case,bairro
8,bairro,NOUN,NOUN,nmod,imóvel
9,Jardim,PROPN,PROPN,appos,bairro


Sintagma(s) Nominal(is) - SN:
 - uma visita
 - imóvel
 - bairro
 - Jardim Europa
 - semana

Sintagma Verbal - SV:
 - Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana

ROOT:
 - A ROOT é 'Gostaria' (VERB). Neste caso, a raiz funciona como verbo principal da oração.
DOCUMENTO 5
A casa na Vila Mariana ainda está disponível para locação?

POS TAGGING:


,Token,POS,Tag,Dependência,Cabeça
0,A,DET,DET,det,casa
1,casa,NOUN,NOUN,nsubj,disponível
2,na,ADP,ADP,case,Vila
3,Vila,PROPN,PROPN,nmod,casa
4,Mariana,PROPN,PROPN,flat:name,Vila
5,ainda,ADV,ADV,advmod,disponível
6,está,AUX,AUX,cop,disponível
7,disponível,ADJ,ADJ,ROOT,disponível
8,para,ADP,ADP,case,locação
9,locação,NOUN,NOUN,obl,disponível


Sintagma(s) Nominal(is) - SN:
 - A casa
 - Vila Mariana
 - locação

Sintagma Verbal - SV:
 - ainda está disponível para locação

ROOT:
 - A ROOT é 'disponível' (ADJ). Há uma cópula ligada à raiz, portanto é uma construção copulativa.



**Comentário:** no Documento 3 a raiz tende a ser um verbo principal ligado à intenção de agendar uma visita.  
No Documento 5, a construção com **“está disponível”** é um caso copulativo: o verbo *estar* liga o sujeito a uma característica/estado, e o parser pode considerar o predicativo como núcleo sintático da oração.



# 4. Interpretação Semântica

São comparados dois pares de palavras presentes no corpus:

1. **apartamento × imóvel**
2. **financiamento × aluguel**


In [8]:

pares = [
    ("apartamento", "imóvel"),
    ("financiamento", "aluguel")
]

for palavra1, palavra2 in pares:
    token1 = nlp(palavra1)[0]
    token2 = nlp(palavra2)[0]

    similaridade = token1.similarity(token2)

    print(f"{palavra1} x {palavra2}: {similaridade:.4f}")


apartamento x imóvel: 0.6667
financiamento x aluguel: 0.3090



**Interpretação:**

- **apartamento × imóvel:** espera-se uma similaridade relativamente alta, porque *apartamento* é um tipo de *imóvel*. Portanto, os conceitos estão semanticamente próximos.
- **financiamento × aluguel:** ambos aparecem no domínio imobiliário, mas representam operações diferentes. Assim, é razoável que a similaridade seja menor do que no primeiro par.

Os valores não precisam ser 1, pois o método trabalha com proximidade entre vetores semânticos aprendidos a partir de grandes coleções de texto.



# 5. Extração de Features

Nesta etapa o corpus inteiro é lematizado e limpo. Depois são construídas:

- Matriz **Bag-of-Words**
- Matriz **TF-IDF**


In [9]:

def preprocessar(texto):
    doc = nlp(texto)

    termos = [
        token.lemma_.lower()
        for token in doc
        if not token.is_space
        and not token.is_punct
        and not token.is_stop
        and token.is_alpha
    ]

    return " ".join(termos)

corpus_lematizado = [preprocessar(texto) for texto in corpus]

for i, texto in enumerate(corpus_lematizado, start=1):
    print(f"Doc {i}: {texto}")


Doc 1: dia nome marcos ferreira interesse apartamento anunciar rua voluntários pátria matão
Doc 2: trabalhar financiamento caixa econômica federal imóvel usar
Doc 3: gostaria agendar visita imóvel bairro jardim europa semana
Doc 4: condomínio iptu apartamento centro araraquara
Doc 5: casa vila mariana disponível locação
Doc 6: aceitam fiador seguro fiança contrato aluguel
Doc 7: corretora camila souza entrar contato interesse cliente joão pedro imóvel ribeirão preto
Doc 8: apartamento reformar recentemente
Doc 9: prazo médio aprovação financiamento bancário
Doc 10: buscar imóvel shopping iguatemi compra vista
Doc 11: sobrado bairro santa cecília aceitar animal estimação
Doc 12: poder enviar foto apartamento planta baixo
Doc 13: taxa juro aplicar banco brasil financiamento
Doc 14: casa alugar vaga garagem cobrir
Doc 15: interesse investir imóvel locação longo prazo
Doc 16: poderia indicar imóvel disponível região central matão


### 5.1 Bag-of-Words

In [10]:

vectorizer_bow = CountVectorizer()
X_bow = vectorizer_bow.fit_transform(corpus_lematizado)

df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer_bow.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(1, len(corpus) + 1)]
)

df_bow


,aceitam,aceitar,agendar,alugar,aluguel,animal,anunciar,apartamento,aplicar,aprovação,araraquara,bairro,baixo,banco,bancário,brasil,buscar,caixa,camila,casa,cecília,central,centro,cliente,cobrir,compra,condomínio,contato,contrato,corretora,dia,disponível,econômica,entrar,enviar,estimação,europa,federal,ferreira,fiador,fiança,financiamento,foto,garagem,gostaria,iguatemi,imóvel,indicar,interesse,investir,iptu,jardim,joão,juro,locação,longo,marcos,mariana,matão,médio,nome,pedro,planta,poder,poderia,prazo,preto,pátria,recentemente,reformar,região,ribeirão,rua,santa,seguro,semana,shopping,sobrado,souza,taxa,trabalhar,usar,vaga,vila,visita,vista,voluntários
Doc 1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Doc 2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0
Doc 3,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0
Doc 4,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Doc 6,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
Doc 7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
Doc 8,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 9,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Doc 10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0


### 5.2 TF-IDF

In [11]:

vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(corpus_lematizado)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer_tfidf.get_feature_names_out(),
    index=[f"Doc {i}" for i in range(1, len(corpus) + 1)]
)

df_tfidf.round(3)


,aceitam,aceitar,agendar,alugar,aluguel,animal,anunciar,apartamento,aplicar,aprovação,araraquara,bairro,baixo,banco,bancário,brasil,buscar,caixa,camila,casa,cecília,central,centro,cliente,cobrir,compra,condomínio,contato,contrato,corretora,dia,disponível,econômica,entrar,enviar,estimação,europa,federal,ferreira,fiador,fiança,financiamento,foto,garagem,gostaria,iguatemi,imóvel,indicar,interesse,investir,iptu,jardim,joão,juro,locação,longo,marcos,mariana,matão,médio,nome,pedro,planta,poder,poderia,prazo,preto,pátria,recentemente,reformar,região,ribeirão,rua,santa,seguro,semana,shopping,sobrado,souza,taxa,trabalhar,usar,vaga,vila,visita,vista,voluntários
Doc 1,0.000,0.000,0.000,0.000,0.000,0.000,0.318,0.225,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.318,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.318,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.248,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.318,0.000,0.277,0.000,0.318,0.000,0.000,0.000,0.000,0.000,0.000,0.318,0.000,0.000,0.000,0.000,0.318,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.318
Doc 2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.409,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.409,0.000,0.000,0.000,0.000,0.409,0.000,0.000,0.000,0.319,0.000,0.000,0.000,0.000,0.246,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.409,0.409,0.000,0.000,0.000,0.000,0.000
Doc 3,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375,0.000,0.225,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.375,0.000,0.000
Doc 4,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.334,0.000,0.000,0.471,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.471,0.000,0.000,0.000,0.471,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.471,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc 5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.421,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.421,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.421,0.000,0.000,0.484,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.484,0.000,0.000,0.000
Doc 6,0.408,0.000,0.000,0.000,0.408,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.408,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.408,0.408,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.408,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Doc 7,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.302,0.000,0.000,0.000,0.000,0.302,0.000,0.000,0.000,0.30

### 5.3 Análise das features

In [12]:

# Documento com maior número de palavras únicas
qtd_unicas = np.asarray((X_bow > 0).sum(axis=1)).ravel()
indice_mais_unicas = int(np.argmax(qtd_unicas))

print(
    f"Documento com mais palavras únicas: Doc {indice_mais_unicas + 1} "
    f"({qtd_unicas[indice_mais_unicas]} termos únicos)"
)
print(corpus[indice_mais_unicas])

# Maior peso TF-IDF da matriz
tfidf_array = X_tfidf.toarray()
linha, coluna = np.unravel_index(np.argmax(tfidf_array), tfidf_array.shape)

termo_maior_tfidf = vectorizer_tfidf.get_feature_names_out()[coluna]
peso_maior_tfidf = tfidf_array[linha, coluna]

print("\nMaior peso TF-IDF encontrado:")
print(f"Documento: Doc {linha + 1}")
print(f"Palavra: {termo_maior_tfidf}")
print(f"Peso: {peso_maior_tfidf:.4f}")
print("Texto:", corpus[linha])

print(
    f"\nComentário: o termo '{termo_maior_tfidf}' recebe peso alto porque é "
    "importante neste documento e aparece pouco ou nada nos demais documentos do corpus."
)


Documento com mais palavras únicas: Doc 7 (12 termos únicos)
Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.

Maior peso TF-IDF encontrado:
Documento: Doc 8
Palavra: recentemente
Peso: 0.6323
Texto: O apartamento é novo ou já foi reformado recentemente?

Comentário: o termo 'recentemente' recebe peso alto porque é importante neste documento e aparece pouco ou nada nos demais documentos do corpus.



**Comentário:** Bag-of-Words representa a frequência absoluta dos termos, enquanto TF-IDF reduz o peso de palavras muito comuns no corpus e destaca termos mais específicos de cada documento.



# 6. Descoberta de Conhecimento em Textos (KDT)

Nesta etapa são aplicadas:

1. NER em documentos com pessoas, locais e organizações;
2. extração de palavras-chave por TF-IDF;
3. LDA com `n_components=2`.


### 6.1 Named Entity Recognition — NER

In [13]:

indices_ner = [0, 1, 6]

for indice in indices_ner:
    doc = docs[indice]

    print("=" * 100)
    print(f"Documento {indice + 1}: {corpus[indice]}")
    print("Entidades encontradas:")

    if doc.ents:
        for ent in doc.ents:
            print(f" - {ent.text:<35} -> {ent.label_}")
    else:
        print(" - Nenhuma entidade reconhecida pelo modelo.")


Documento 1: Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.
Entidades encontradas:
 - Marcos Ferreira                     -> PER
 - Rua Voluntários da Pátria           -> LOC
 - Matão                               -> LOC
Documento 2: Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?
Entidades encontradas:
 - Vocês                               -> LOC
 - Caixa Econômica Federal             -> ORG
Documento 7: Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.
Entidades encontradas:
 - Camila Souza                        -> PER
 - João Pedro                          -> PER
 - Ribeirão Preto                      -> LOC



**Comentário:** o modelo reconheceu corretamente entidades como Marcos Ferreira, Camila Souza e João Pedro como pessoas (PER), Matão, Rua Voluntários da Pátria e Ribeirão Preto como locais (LOC) e Caixa Econômica Federal como organização (ORG). Porém, o termo “Vocês” foi classificado incorretamente como local (LOC), mostrando que modelos de NER podem gerar falsos positivos e que os resultados precisam ser analisados, e não apenas aceitos automaticamente.


### 6.2 Palavras-chave de cada documento com TF-IDF

In [14]:

termos_tfidf = vectorizer_tfidf.get_feature_names_out()

def top_palavras_documento(linha_tfidf, quantidade=5):
    valores = linha_tfidf.toarray().ravel()
    indices = valores.argsort()[::-1]
    indices = [i for i in indices if valores[i] > 0][:quantidade]

    return [
        (termos_tfidf[i], valores[i])
        for i in indices
    ]

for i in range(len(corpus)):
    palavras = top_palavras_documento(X_tfidf[i], quantidade=5)

    print(f"Doc {i + 1}:")
    print(", ".join(f"{termo} ({peso:.3f})" for termo, peso in palavras))
    print()


Doc 1:
voluntários (0.318), pátria (0.318), rua (0.318), dia (0.318), ferreira (0.318)

Doc 2:
trabalhar (0.409), usar (0.409), federal (0.409), econômica (0.409), caixa (0.409)

Doc 3:
visita (0.375), semana (0.375), jardim (0.375), agendar (0.375), europa (0.375)

Doc 4:
centro (0.471), araraquara (0.471), iptu (0.471), condomínio (0.471), apartamento (0.334)

Doc 5:
vila (0.484), mariana (0.484), locação (0.421), disponível (0.421), casa (0.421)

Doc 6:
seguro (0.408), fiança (0.408), aluguel (0.408), contrato (0.408), aceitam (0.408)

Doc 7:
preto (0.302), ribeirão (0.302), souza (0.302), corretora (0.302), entrar (0.302)

Doc 8:
reformar (0.632), recentemente (0.632), apartamento (0.448)

Doc 9:
bancário (0.479), aprovação (0.479), médio (0.479), prazo (0.417), financiamento (0.373)

Doc 10:
vista (0.432), shopping (0.432), iguatemi (0.432), compra (0.432), buscar (0.432)

Doc 11:
sobrado (0.385), santa (0.385), estimação (0.385), cecília (0.385), aceitar (0.385)

Doc 12:
enviar (


**Comentário:** as palavras-chave representam os termos que melhor diferenciam cada documento do restante do corpus. Termos muito específicos, como nomes de bairros, bancos ou tipos de negociação, tendem a obter pesos elevados.


### 6.3 LDA — 2 tópicos

In [15]:

lda = LatentDirichletAllocation(
    n_components=2,
    random_state=42,
    learning_method="batch"
)

lda.fit(X_bow)

termos_bow = vectorizer_bow.get_feature_names_out()

for numero_topico, pesos in enumerate(lda.components_, start=1):
    indices = pesos.argsort()[-10:][::-1]
    palavras = [termos_bow[i] for i in indices]

    print(f"Tópico {numero_topico}:")
    print(", ".join(palavras))
    print()


Tópico 1:
imóvel, interesse, apartamento, bairro, preto, ribeirão, souza, pedro, joão, corretora

Tópico 2:
imóvel, apartamento, financiamento, matão, disponível, casa, locação, prazo, bairro, rua



In [16]:

distribuicao = lda.transform(X_bow)

df_lda = pd.DataFrame({
    "Documento": [f"Doc {i}" for i in range(1, len(corpus) + 1)],
    "Trecho": [texto[:70] + ("..." if len(texto) > 70 else "") for texto in corpus],
    "Tópico 1": distribuicao[:, 0],
    "Tópico 2": distribuicao[:, 1]
})

df_lda["Tópico predominante"] = np.argmax(distribuicao, axis=1) + 1

df_lda.round({
    "Tópico 1": 3,
    "Tópico 2": 3
})


,Documento,Trecho,Tópico 1,Tópico 2,Tópico predominante
0,Doc 1,Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento a...,0.048,0.952,2
1,Doc 2,Vocês trabalham com financiamento pela Caixa Econômica Federal para im...,0.070,0.930,2
2,Doc 3,Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta...,0.063,0.937,2
3,Doc 4,Qual é o valor do condomínio e do IPTU do apartamento no centro de Ara...,0.092,0.908,2
4,Doc 5,A casa na Vila Mariana ainda está disponível para locação?,0.092,0.908,2
5,Doc 6,Aceitam fiador ou seguro fiança para o contrato de aluguel?,0.926,0.074,1
6,Doc 7,"Sou a corretora Camila Souza, entrando em contato sobre o interesse do...",0.958,0.042,1
7,Doc 8,O apartamento é novo ou já foi reformado recentemente?,0.853,0.147,1
8,Doc 9,Qual o prazo médio de aprovação do financiamento bancário?,0.096,0.904,2
9,Doc 10,"Estou buscando um imóvel próximo ao Shopping Iguatemi, para compra à v...",0.922,0.078,1



**Interpretação do LDA:** como o corpus é pequeno e foram definidos apenas dois tópicos, a separação não é totalmente clara. O Tópico 1 destacou termos como imóvel, interesse, apartamento, bairro, Ribeirão Preto, Souza, Pedro, João e corretora, ficando bastante influenciado por mensagens relacionadas a clientes, interesse em imóveis e informações específicas de localização. Já o Tópico 2 apresentou termos como imóvel, apartamento, financiamento, Matão, disponível, casa, locação e prazo, concentrando mais mensagens relacionadas a financiamento, disponibilidade e condições dos imóveis. A distribuição mostra que cada documento possui uma probabilidade para os dois tópicos, sendo considerado predominante aquele com maior valor.



# 7. Conclusão Integradora

As etapas iniciais do pipeline foram pré-requisitos para a descoberta de conhecimento porque transformaram o texto bruto em uma representação estruturada e comparável. A tokenização separou cada documento em unidades que puderam ser filtradas e analisadas; a remoção de stopwords e a lematização reduziram ruído e unificaram variações de uma mesma palavra; e a análise sintática e semântica ajudou a compreender o papel e a relação entre os termos. Um exemplo concreto deste notebook é a construção das matrizes Bag-of-Words e TF-IDF: antes de gerar essas features, o corpus foi lematizado e palavras pouco informativas foram removidas. Isso permitiu que termos realmente característicos de cada mensagem recebessem mais destaque no TF-IDF e, posteriormente, fossem usados tanto para extrair palavras-chave quanto para alimentar o LDA. Dessa forma, as técnicas iniciais não são etapas isoladas, mas formam a preparação necessária para que os métodos de KDT consigam encontrar padrões e tópicos no conjunto de textos.
